# Передача данных по сети

# Даг Хеллман. Стандартная библиотека Python 3
* [ ] Глава 11. Обмен данными по сети 695
    * [x] 11.1. ~~`ipaddress`: интернет-адреса 695~~
    * [ ] 11.2. `socket`: сетевое взаимодействие 701
    * [ ] 11.3. `selectors`: абстракции мультиплексирования ввода-вывода 731
    * [ ] 11.4. `select`: эффективное ожидание завершения ввода-вывода 736
    * [ ] 11.5. `socketserver`: создание сетевых серверов 749

* [ ] Глава 12. Интернет 761
    * [ ] 12.1. `urllib.parse`: разбиение URL-адресов на отдельные элементы 762
    * [ ] 12.2. `urllib.request`: доступ к сетевым ресурсам 769
    * [ ] 12.3. `urllib.robotparser`: управление действиями веб-роботов 780
    * [ ] 12.4. `base64`: кодирование двоичных данных с помощью ASCII 783
    * [ ] 12.5. `http.server`: базовые классы для реализации веб-серверов 788
    * [ ] 12.6. `http.cookies`: cookie-файлы HTTP 796
    * [ ] 12.7. `webbrowser`: отображение веб-страниц 801
    * [ ] 12.8. `uuid`: универсальные уникальные идентификаторы 803
    * [ ] 12.9. `json`: JavaScript Object Notation 809
    * [ ] 12.10. `xmlrpc.client`: клиент XML-RPC 821
    * [ ] 12.11. `xmlrpc.server`: сервер XML-RPC 832

## `ipaddress`: интернет-адреса

Библиотека ipaddress включает классы, которые обеспечивают сравнение и другие операции над сетевыми адресами IPv4 и IPv6.

### Проверка на версию IP (4 или 6), а также различные представления

In [6]:
# ipaddress__addresses.py
import binascii # библиотека для конвертации между двоичным кодом и ASCII
import ipaddress

adresses = [
    '10.9.0.6',
    'fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa'
]

for ip in adresses:
    addr = ipaddress.ip_address(ip) # Используя данную функцию не надо заботится о детекции версии IP, все IP будут иметь 1 универсальное представление. 
    print('{!r}'.format(addr))
    print(' IP version:', addr.version) # версия IP, нужно для раздельной обработки протоколов на уровне приложения.
    print(' is private:', addr.is_private) # Проверяем используется ли IP в частной сети.
    print('packed form:', binascii.hexlify(addr.packed)) # компактное бинарное представление и преобразует в читаемую 16-ричную форму.
    print(' integer:', int(addr))
    print()

IPv4Address('10.9.0.6')
 IP version: 4
 is private: True
packed form: b'0a090006'
 integer: 168361990

IPv6Address('fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa')
 IP version: 6
 is private: True
packed form: b'fdfd87b5b4755e3eb1bce121a8eb14aa'
 integer: 337611086560236126439725644408160982186



### Сети

Сети поределяются диапозоном алресов. Обычно это базовый адрес + маска сети, указывающие какие части адреса представляют сеть, а какие - адреса хостов в этой сети. 
Маска может быть выражена явно, либо системой обозначений "префикс/число битов"

#### Анализ сети IP

In [17]:
import ipaddress

NETWORKS = [
    '10.9.0.0/24',
    'fdfd:87b5:b475:5e3e::/64'
]
for item in NETWORKS:
    net = ipaddress.ip_network(item)
    print('{!r}'.format(net))
    print(f'     is private:{net.is_private}')
    print(f'     broadcast:{net.broadcast_address}')
    print(f'     compressed:{net.compressed}')
    print(f'     with netmask:{net.with_netmask}')
    print(f'     with hostmas:{net.with_hostmask}')
    print(f'     num address:{net.num_addresses}')
    print('      first 3 ip in the network:')
    for i, ip in zip(range(3), net):
        print(f' {i}     {ip}')
    print('      first 3 hosts in the network:')
    for i, ip in zip(range(3), net.hosts()):
        print(f' {i}     {ip}')
    print()

IPv4Network('10.9.0.0/24')
     is private:True
     broadcast:10.9.0.255
     compressed:10.9.0.0/24
     with netmask:10.9.0.0/255.255.255.0
     with hostmas:10.9.0.0/0.0.0.255
     num address:256
      first 3 ip in the network:
 0     10.9.0.0
 1     10.9.0.1
 2     10.9.0.2
      first 3 hosts in the network:
 0     10.9.0.1
 1     10.9.0.2
 2     10.9.0.3

IPv6Network('fdfd:87b5:b475:5e3e::/64')
     is private:True
     broadcast:fdfd:87b5:b475:5e3e:ffff:ffff:ffff:ffff
     compressed:fdfd:87b5:b475:5e3e::/64
     with netmask:fdfd:87b5:b475:5e3e::/ffff:ffff:ffff:ffff::
     with hostmas:fdfd:87b5:b475:5e3e::/::ffff:ffff:ffff:ffff
     num address:18446744073709551616
      first 3 ip in the network:
 0     fdfd:87b5:b475:5e3e::
 1     fdfd:87b5:b475:5e3e::1
 2     fdfd:87b5:b475:5e3e::2
      first 3 hosts in the network:
 0     fdfd:87b5:b475:5e3e::1
 1     fdfd:87b5:b475:5e3e::2
 2     fdfd:87b5:b475:5e3e::3



Метод `is_private` позволяет быстро понять, внутренняя это подсеть или публичная.
Применяется:
* при фильтрации трафика (разрешить/запретить доступ к внутренним ресурсам),
* при аудите безопасности — чтобы убедиться, что сервис не «светит» публичные адреса там, где не должен,
* в скриптах миграции, когда надо отделить офисные сети от облачных.



Итерация `for ip in net` перебирает все адреса подсети, включая `network` и `broadcast` (для IPv4).

Итерация `net.hosts()` — только хосты (без `network` и `broadcast`).

Применение:
* Раздача IP‑адресов новым виртуалкам/контейнерам (например, выбираем следующий свободный),
* Пакетное сканирование сети (ping sweep, проверка открытых портов) — генерируем список всех хостов,
* Настройка DHCP-пула — можно программно получить диапазон адресов, доступных для выдачи клиентам,
* Мониторинг — обойти все хосты подсети и собрать метрики.

`num_addresses` сразу показывает, сколько узлов может быть в сети. Для IPv6 это очень большое число, но для IPv4 — помогает оценить, хватит ли подсети для растущего количества устройств.

##### ▶ Подсеть (IP network) и CIDR-нотация

Запись вида `10.9.0.0/24` или `fdfd:87b5:b475:5e3e::/64` задаёт диапазон адресов (*подсеть*).

Число после косой черты — **префикс** (длина маски в битах). Оно показывает, сколько первых бит адреса зафиксированы как *«адрес сети»*, а остальные биты отданы под нумерацию хостов.
* `/24` в IPv4 означает, что первые 24 бита неизменны, меняются только последние 8 бит — всего 256 адресов.
* `/64` в IPv6 — фиксированы первые 64 бита, под хосты отдано 64 бита — колоссальное количество адресов.

##### ▶ Приватный адрес / подсеть (`is_private`)
Диапазоны адресов, зарезервированные для использования внутри локальных сетей (дом, офис), не маршрутизируются в публичном интернете.

Примеры:
* IPv4: 10.0.0.0/8, 192.168.0.0/16, 172.16.0.0/12
* IPv6: fd00::/8 (Unique Local Addresses)

Метод `is_private` возвращает `True`, если подсеть (или адрес) принадлежит таким диапазонам. Удобно для быстрой проверки «это локальный адрес или публичный?»

##### ▶ Широковещательный адрес (`broadcast_address`)
Специальный адрес в IPv4-подсети, по которому пакет доставляется всем устройствам этой подсети сразу.
Например, в сети `10.9.0.0/24` это `10.9.0.255`. Устройства не могут использовать его как собственный адрес.

В IPv6 концепция широковещания отсутствует (заменена групповой рассылкой), но библиотека `ipaddress` для единообразия всё равно возвращает последний адрес диапазона — ffff:ffff:ffff:... — хотя практически он как широковещательный не используется.

##### ▶ Маска подсети (`with_netmask`)
Показывает тот же префикс, но в виде отдельного адреса, где единицы задают часть сети, а нули — часть хоста.
Для `10.9.0.0/24` это `255.255.255.0`. Метод `with_netmask` возвращает строку вида IP_адрес/маска (например, `10.9.0.0/255.255.255.0`).

##### ▶ Обратная маска (hostmask) (`with_hostmask`)
Она же «маска хостов», *wildcard mask*. Получается инвертированием сетевой маски (биты хостовой части — единицы).
Используется в некоторых протоколах (например, в списках доступа Cisco). Для той же сети будет `0.0.0.255`. Вывод: `10.9.0.0/0.0.0.255`.

##### ▶ Сжатое представление (`compressed`)
Актуально для IPv6. Длинную запись можно сократить:
* отбросить ведущие нули в каждой группе,
* самую длинную последовательность нулевых групп заменить на ::.

Например,
`fdfd:87b5:b475:5e3e:0000:0000:0000:0000` превращается в `fdfd:87b5:b475:5e3e::`.

Метод `compressed` всегда даёт короткую, читаемую форму адреса сети.

##### ▶ Количество адресов в сети (`num_addresses`)
Сколько всего IP-адресов входит в данную подсеть (включая сетевой адрес, широковещательный и все хостовые).

* Для `10.9.0.0/24` — 256 адресов (от 0 до 255).
* Для `fdfd:87b5:b475:5e3e::/64` — 2⁶⁴ (~18 квинтиллионов) адресов.

##### ▶ Итерация по сети и метод `hosts()`
Когда пишем `for ip in net`, Python перебирает все адреса подряд, от первого (сетевого) до последнего (широковещательного).
Метод `net.hosts()` возвращает итератор только по тем адресам, которые можно назначить устройствам:
* для IPv4 исключены сетевой адрес (например, `10.9.0.0`) и широковещательный (`10.9.0.255`);
* для IPv6 исключён только сетевой адрес (Subnet-Router anycast address).

Это удобно, если нужно перебрать только «живые» адреса для раздачи или сканирования, не задевая служебные.

##### Проверка вхождения IP в сеть 

In [ ]:
import ipaddress
NETWORKS = [
    ipaddress.ip_network('10.9.0.0/24'),
    ipaddress.ip_network('fdfd:87b5:b475:5e3e::/64'),
]
ADDRESSES = [
    ipaddress.ip_address ('10.9.0.6'),
    ipaddress.ip_address('10.7.0.31'),
    ipaddress.ip_address('fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa'),
    ipaddress.ip_address('fe80::3840:c439:b25e:63b0'),
]
for ip in ADDRESSES:
    for net in NETWORKS:
        if ip in net:
            print('{}\nis on {}'.format(ip, net))
            break
    else:
        print('{}\nis not on a known network'.format(ip))
    print()

10.9.0.6
is on 10.9.0.0/24

10.7.0.31
is not on a known network

fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa
is on fdfd:87b5:b475:5e3e::/64

fe80::3840:c439:b25e:63b0
is not on a known network



Класс сети позволяет оператор вхождения `in`.

### Интерфейсы

Сетевой интерфейс - это особый сетевой адрес, который может быть представлен в виде адреса хоста и сетевого префикса или макси сети.

In [21]:
import ipaddress
ADDRESSES = [
    '10.9.0.6/24',
    'fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa/64',
]
for ip in ADDRESSES:
    iface = ipaddress.ip_interface(ip)
    print('{!r}'.format(iface))
    print('network:\n ', iface.network)
    print('ip:\n ', iface.ip)
    print('IP with prefixlen:\n ', iface.with_prefixlen)
    print('netmask:\n ', iface.with_netmask)
    print ('hostmask:\n ', iface.with_hostmask)
    print()

IPv4Interface('10.9.0.6/24')
network:
  10.9.0.0/24
ip:
  10.9.0.6
IP with prefixlen:
  10.9.0.6/24
netmask:
  10.9.0.6/255.255.255.0
hostmask:
  10.9.0.6/0.0.0.255

IPv6Interface('fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa/64')
network:
  fdfd:87b5:b475:5e3e::/64
ip:
  fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa
IP with prefixlen:
  fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa/64
netmask:
  fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa/ffff:ffff:ffff:ffff::
hostmask:
  fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa/::ffff:ffff:ffff:ffff



## `socket`: сетевое взаимодействие

Низкоуровневая библиотека `socket` прдоставляет непосредственный доступ к C-библиотеке `socket` для взаимодейстий по сети с использование BSD-сокетов. Класс `socket` предназначен для работы с фактическим канадлом передачи данных, а также функции для решения задач: переименование сервера в адрес, форматирование данных передаваемых по сети

### Что такое сокет?

**Сокет** — это конечная точка соединения, используемого программами как для локального обмена данными между процессами, так и для обмена через Интернет.
За управление процессом передачи данных отвечают два основных свойства сокетов: 
* **семейство адресов**, задающее используемый сетевой протокол модели OSI
* **тип сокета**, соответствующий протоколу транспортного уровня.

Python поддерживает по крайней мере три семейства адресов. Среди наиболее распространенных из них:


### Семейства адресов


##### Семейство адресов `AF_INET`
Данное семейство используется для IРv4-адресации в Интернете. Адреса IPv4 имеют длину 4 байта и обычно представляются в виде последовательности из четырех чисел, по одному на октет, разделенных точками (например, `10.1.1.5` или `127.0.0.1`). Именно такие значения чаще всего имеют в виду, когда говорят об IP-адресах. В настоящее время почти вся работа в Интернете ведется c использованием адресов IPv4.

##### Семейство адресов `AF_INET6`
Семейство адресов `AF_INET6` используется для IРv6-адресации в Интернете.
Протокол IPv6 — это следующее поколение интернет-протокола, обеспечивающее поддержку 128-битовых адресов, шейпинг трафика и средства маршрутизации, недоступные в протоколе IPv4. Внедрение 1Ру6-адресации идетускоренными темпами, особенно c появлением облачных технологий и добавлением в сеть новых устройств в рамках проектов интернета вещей.

##### Семейство адресов `AF_UNIX`
`AF_UNIX` применяется при использовании сокетов домена Unix (Unix Domain Sockets — UDS) — протокола межпроцессного взаимодействия, доступного в POSIX-совместимых системах. В типичных случаях реализация UDS
позволяет операционной системе передавать данные непосредственно из процесса в процесс без прохождения сетевого стека. Такой подход обеспечивает бОльшую эффективность по сравнению c использованием `AF_INET`, но ввиду того, что в качестве пространства имен для адресации используется файловая система, применение UDS ограничено процессами, выполняющимися в одной и той же
системе. Привлекательность использования UDS по сравнению c другими механизмами межпроцессного взаимодействия, такими как именованные каналы или разделяемая память, заключается в том, что программный интерфейс остается тем же, что и при работе c IP-сетями, благодаря чему приложение может воспользоваться всеми преимуществами эффективной передачи данных при выполнении программы на отдельном хосте, но при этом использовать тот же код для передачи данных по сети.

Константа `AF_UNIX` определена лишь в системах, поддерживающих UDS.

#### Типы сокетов

Обычно типом сокета является:
* либо `SOCK_DGRAM`, предназначенный для передачи датаграмм без предварительного установления соединения
* либо `SOCK_STREAM`, предназначенный для передачи потока байтов c поддержкой логического соединения. 
* Сокеты, работающие c датаграммами, чаще всего ассоциируются c протоколом пользовательских датаграмм (User Datagram Protocol — `UDP`). Сокеты этого типа не в состоянии обеспечить высокую надежность передачи данных.
* Потокоориентированные сокеты типа `SOCK_STREAM` ассоциируются c протоколом,
управления, передачей (Transport Control Protocol — `TCP`). Они обеспечивают передачу байтовых потоков между сервером и клиентом и гарантируют доставку сообщений или уведомлений о неудачной попытке доставки, используя управление тайм-аутами, повторную передачуданных и другие возможности.

Большинство протоколов прикладного уровня, обеспечивающих доставку больших объемов данных, такие как HTTP, располагаются в стеке поверх протокола TCP, поскольку создание сложных приложений упрощается, если упорядочение и доставка сообщений осуществляются автоматически. Обычно UDP используют при работе c протоколами, в которых порядок получения сообщений не особенно существен (поскольку каждое сообщение является самодостаточным и имеет небольшой размер, как, например, в случае получения имени хоста c помощью службы DNS), а также при многоадресатном вещании (когда одни и те же данные передаются нескольким хостам). Оба протокола, UDP и TCP, могут использоваться как c IPv4-, так и c IРv6-адресацией.

Модуль `socket` в Python поддерживает и другие типы сокетов, но они используются не так часто и поэтому здесь не рассматриваются. Для получения более подробной информации по этому вопросу обратитесь к документации стандартной библиотеки.

### Поиск хостов в сети

Модуль `socket` включает функции, образующие программный интерфейс к сетевым службам доменных имен, благодаря чему программа может преобразовать имя серверного хоста в числовой сетевой адрес. Приложениям не нужно выполнять явное преобразование адресов до того, как они будут использованы для установления соединения c сервером. Тем не менее при выводе сообщений об ошибках в них целесообразно включать как числовой адрес, так и используемое имя сервера.

#### Имя локального компьютера

In [22]:
import socket
print(socket.gethostname())

PC301


#### IP вместо имени сервера

In [26]:
import socket
HOSTS = [
    'PC301',
    'pymotw.com',
    'www.python.org',
    'nosuchname'
]
for host in HOSTS:
    try:
        print('{} : {}'.format(host, socket.gethostbyname(host)))
    except socket.error as msg:
        print('{} : {}'.format(host, msg))

PC301 : 127.0.1.1
pymotw.com : 185.199.111.153
www.python.org : 151.101.128.223
nosuchname : [Errno -5] No address associated with hostname


Как мы видим, даже наше локальное устройство было идентифицировано с loopback-ip (нужен, чтобы отправлять запросы "самому себе").
Если указанное имя не удается найти, то возбуждается `socket.error`

Для получения более подробной информации используется `gethostbyname_ex(host)`

In [30]:
import socket
HOSTS = [
    'PC301',
    'pymotw.com',
    'www.python.org',
    'nosuchname'
]
for host in HOSTS:
    print(host)
    try:
        name, aliases, addresses = socket.gethostbyname_ex(host)
        print(' Hostname:', name)
        print(' Aliases :', aliases)
        print(' Addresses:', addresses)
    except socket.error as msg:
        print('ERROR:', msg)
    print()

PC301
 Hostname: PC301
 Aliases : []
 Addresses: ['127.0.1.1']

pymotw.com
 Hostname: pymotw.com
 Aliases : []
 Addresses: ['185.199.109.153', '185.199.108.153', '185.199.111.153', '185.199.110.153']

www.python.org
 Hostname: dualstack.python.map.fastly.net
 Aliases : ['www.python.org']
 Addresses: ['151.101.64.223', '151.101.192.223', '151.101.0.223', '151.101.128.223']

nosuchname
ERROR: [Errno -5] No address associated with hostname



## `selectors`: абстракции мультиплексирования ввода-вывода

Модуль `selectors` предоставляет высокоуровневый интерфейс для одновременного наблюденияза несколькими сокетами и обеспечивает возможность взаимодействия сетевых серверов с несколькими клиентами одновременно. 

## `select`: эффективное ожидание ввода-вывода

Предоставляет низкоуровневые программные интерфейсы, используемые модулем `selectors`.

## `socketserver`: создание сетевых серверов

Фреймворки в составе `socketsetvers`, абстрагируют значительную часть рутинной работы, которую приходится выполянть при создании нового сетевого сервера. 

# Интернет

## `urlib.parse`: разбиение URL-адресмов на отдельные элементы

## `urlib.request`: доступ к сетевым ресурсам

## `urlib.robotparser`: управление действиями веб-роботов

## `base64`: кодирование двоичных данных с помощью ASCII

## `http.server`: базовые классы для реализации веб-серверов

## `http.cookies`: cookie-файлы HTTP

## `webbrowser`: отображение веб-страниц

## `uuid`: универсальные уникальные идентифкаторы

## `json`: JavaScript Object Notation

## `xmlrpc.client`: клиент XML-RPC

## `xmlrpc.server`: сервер XML-RPC

# Электронная почта